# Coop Case — Q4.1: How Will the 2026 Election Affect Coop's Business?

**Question:** Do the math on taxes -- what do the top parties say they'll do with taxes, and how
would that affect Coop's business? Scoped to the top 3 parties by current polling.

**Framing correction:** Sweden is a parliamentary democracy (Riksdag), not a presidential system --
there's no "presidency" race. The relevant event is the general election on **2026-09-13**, which
elects the 349 Riksdag seats. As of writing (session date 2026-08-31/09-01), that election is about
two weeks away.

**Top 3 parties by current polling** (source: PolitPro trend, Aug 30 2026; Wikipedia opinion-polling page):
1. Socialdemokraterna (S) — ~33%
2. Sverigedemokraterna (SD) — ~19%
3. Moderaterna (M) — ~17-18% (the current PM's party)


## 0. Sources & method

This notebook combines two things:
1. **Real, cited research** on the three parties' tax positions (via web search -- not fabricated)
2. **Quantitative modeling** using the actual 2020 receipt data as a sizing/mechanism illustration

**Important scope caveat:** the receipt data is from **April-May 2020**, and covers only 2 of
Coop's stores. It cannot show a real reaction to the 2026 tax environment -- there's no natural
experiment here (VAT didn't change during the observed window). What it *can* do is show the
**magnitude and mechanism** of how a VAT or corporate-tax change would flow through Coop's actual
revenue/cost structure, using real transaction-level numbers rather than made-up ones. Extrapolating
this 2-store, 2-month sample to Coop's true national exposure would be wrong -- treat all SEK
figures below as illustrative of mechanism and scale, not a forecast of Coop's real P&L.

**Sources** (fetched during this session):
- [2026 Swedish general election — Wikipedia](https://en.wikipedia.org/wiki/2026_Swedish_general_election)
- [Opinion polling for the 2026 Swedish general election — Wikipedia](https://en.wikipedia.org/wiki/Opinion_polling_for_the_2026_Swedish_general_election)
- [Sweden Election Polls 2026 — PolitPro](https://politpro.eu/en/sweden)
- [Sänkt matmoms 2026: 6% på livsmedel — Grant Thornton](https://www.grantthornton.se/insikt/skattenyhet/matmomsen-sanks-men-komplexiteten-okar-sa-navigerar-ratt-fran-dag-ett/)
- [Skatter och partier 2026 — Valkoll](https://valkoll2026.se/insikt/skatter-partier-2026/)
- [Fakta om S politik: skattepolitik — Socialdemokraterna](https://www.socialdemokraterna.se/var-politik/a-till-o/skatter/fakta-om-s-politik-skattepolitik)
- [Skatterna och valet — SD vill sänka skatter för hushåll — PwC](https://blogg.pwc.se/taxmatters/skatterna-valet-sverigedemokraterna)
- [Sänkt skatt för företag — Regeringen.se](https://www.regeringen.se/rattsliga-dokument/departementsserien-och-promemorior/2025/05/sankt-skatt-for-foretag/)
- [Bolagsskatt — internationellt — Ekonomifakta](https://www.ekonomifakta.se/sakomraden/skatt/skatt-pa-foretagande-och-kapital/bolagsskatt-internationellt_1212527.html)


## 1. Party tax positions relevant to a grocery retailer

| | Socialdemokraterna (S), ~33% | Sverigedemokraterna (SD), ~19% | Moderaterna (M), ~17-18% |
|---|---|---|---|
| **Food VAT (matmoms)** | Explicitly proposed a temporary cut to 6% (from 12%) to ease household costs | Not a headline SD proposal specifically, but SD (as a budget-supporting party) claims credit for the ~600 kr/month household saving from the cut | Part of the governing coalition that implemented the cut (Apr 2026-Dec 2027) |
| **Corporate tax (bolagsskatt)** | Wants to **hold at the current 20.6%** level | Not clearly documented in available sources | Wants to cut **below 20%** for international competitiveness (no exact target stated); government has already proposed 20.0% |
| **Other notable proposals** | New bank tax; "beredskapsskatt" (defense-readiness tax); ISK tax-free up to 300k SEK | Lower income & electricity tax; cap top marginal rate at 50%; cut fuel tax | Broadest combined tax cuts among the blocs (per PwC's roundup) |

**Reality check on what's already happened:** the food VAT cut to 6% is not a campaign promise --
it's **already in effect** (since 2026-04-01, running through 2027-12-31) and has support (or at
least no clear opposition) across all three parties. The live political question is what happens
**after** it expires -- and a government-commissioned review of a differentiated food VAT (up to 3
rates) is due by 2026-12-22. Corporate tax is the sharper dividing line: S wants to hold the line,
M wants to go lower, and SD's position isn't clearly staked out in the sources found.


## 2. Setup & load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)


In [ ]:
DATA_PATH = "2months_v2/rl_2months.csv"

dtypes = {
    "receiptKey": "int64",
    "hourOfDay": "int8",
    "minuteOfHour": "int8",
    "quantity": "float32",
    "lineItemAmount": "float32",
    "lineItemAmountExclVat": "float32",
    "discountAmountExclVat": "float32",
    "lineItemCostExclVat": "float32",
    "CoopOnlineYN": "category",
    "store": "category",
    "customerId": "Int64",
    "householdId": "Int64",
    "MOSAICGroup": "category",
    "MOSAICGroupDescription": "category",
    "MOSAICType": "category",
    "MOSAICTypeDescription": "category",
    "DominantBuyingPowerClass": "category",
    "ItemID": "int64",
    "ItemSubSegmentName": "category",
    "ItemSubSegmentID": "Int64",
    "ItemSegmentName": "category",
    "ItemSegmentID": "Int64",
    "ItemSubCategoryName": "category",
    "ItemSubCategoryID": "Int64",
    "ItemCategoryName": "category",
    "ItemCategoryID": "Int64",
    "ItemCategoryTeamName": "category",
    "ItemCategoryTeamID": "Int64",
    "ItemCategoryGroupName": "category",
    "ItemCategoryGroupID": "Int64",
    "ItemCategoryAreaName": "category",
    "ItemCategoryAreaID": "Int64",
    "Brand": "category",
    # Stored as floats in the source file (0.0 / 1.0), not clean ints -- pandas
    # won't safely downcast float64 -> int8 during read_csv, so keep as float32.
    "eko": "float32",
    "organic": "float32",
    "krav": "float32",
    "fair_trade": "float32",
    "msc": "float32",
    "no_lactose": "float32",
}

df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=["DayDate"],
)
df["profit"] = df["lineItemAmountExclVat"] - df["lineItemCostExclVat"]

print(df.shape)
df.head()

## 3. Isolating "food" transactions directly from the data

Rather than guessing which categories were taxed at the food rate, we can **reverse-engineer the
actual VAT rate per line** from `lineItemAmount` (incl. VAT) vs. `lineItemAmountExclVat` (excl. VAT).
Sweden's real VAT brackets (25% standard, 12% food/hotel, 6% books/newspapers, 0% exempt) show up
cleanly in the data -- this is a precise, data-driven way to isolate exactly what the 2026 food VAT
cut (12% -> 6%) would apply to, rather than relying on the (noisier) category-name hierarchy.


In [ ]:
mask = df["lineItemAmountExclVat"] > 1  # avoid tiny/zero denominators and refund edge cases
implied_vat_pct = ((df.loc[mask, "lineItemAmount"] / df.loc[mask, "lineItemAmountExclVat"] - 1) * 100).round(0)

print("Implied VAT rate distribution (line items):")
print(implied_vat_pct.value_counts().sort_index())


## 4. Scenario 1: the food VAT cut (12% -> 6%)

Isolate the 12%-VAT lines (the pre-2026 food rate) and simulate what the 6% rate implies, using
this 2-month sample as the base. Two business scenarios for Coop:
- **Pass-through**: Coop keeps its excl-VAT price the same -> the full VAT cut becomes a lower
  consumer price (a demand/volume lever, not a margin lever)
- **Retain**: Coop raises its excl-VAT price so the consumer-facing price stays roughly the same
  -> Coop captures the VAT-cut difference as extra margin


In [ ]:
food_mask = mask.reindex(df.index, fill_value=False) & (implied_vat_pct.reindex(df.index) == 12)
food = df[food_mask]

excl_vat = food["lineItemAmountExclVat"].sum()
incl_vat_12 = food["lineItemAmount"].sum()
incl_vat_6 = excl_vat * 1.06

print(f"Food (12%-VAT) line items: {food.shape[0]:,}")
print(f"Food revenue excl. VAT:            {excl_vat:,.0f} SEK")
print(f"Consumer price incl. VAT @ 12% (historic): {incl_vat_12:,.0f} SEK")
print(f"Consumer price incl. VAT @ 6%  (simulated): {incl_vat_6:,.0f} SEK")
print()

savings_passthrough = incl_vat_12 - incl_vat_6
print(f"Scenario A -- Pass-through: consumer saves {savings_passthrough:,.0f} SEK over 2 months "
      f"({savings_passthrough/incl_vat_12:.2%} price drop on food)")

new_excl_vat_if_price_held = incl_vat_12 / 1.06
extra_margin = new_excl_vat_if_price_held - excl_vat
print(f"Scenario B -- Retain margin: Coop captures {extra_margin:,.0f} SEK extra over 2 months "
      f"({extra_margin/excl_vat:.2%} revenue uplift on food), consumer price unchanged")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
pd.Series({
    "Historic\n(12% VAT)": incl_vat_12,
    "Pass-through\n(6% VAT, consumer saves)": incl_vat_6,
}).plot(kind="bar", ax=ax, color=["#8FA097", "#22B573"])
ax.set_title("Food VAT cut: consumer-facing price, historic vs. pass-through scenario")
ax.set_ylabel("SEK (2-month sample, both stores)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


## 5. Scenario 2: corporate tax rate

Applies candidate rates to the sample's **gross margin** (`profit` = revenue excl. VAT − cost excl. VAT)
as a rough upper-bound proxy for taxable profit. This is **not** real taxable income -- actual
corporate tax applies after wages, rent, depreciation, interest, etc., none of which are in this
dataset. Treat this purely as a mechanism/sensitivity illustration, scaled to this 2-store sample only.


In [ ]:
total_margin = df["profit"].sum()
rates = {
    "20.6% (status quo)": 20.6,
    "20.0% (government proposal)": 20.0,
    "19.0% (illustrative -- M wants '<20%', no exact figure stated)": 19.0,
}

rows = []
for label, rate in rates.items():
    net = total_margin * (1 - rate / 100)
    rows.append({"scenario": label, "rate_pct": rate, "tax_paid": total_margin - net, "net_of_tax_margin": net})

tax_table = pd.DataFrame(rows).set_index("scenario").round(0)
tax_table


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tax_table["net_of_tax_margin"].plot(kind="bar", ax=ax, color="#22B573")
ax.set_title("Net-of-tax gross margin under candidate corporate tax rates\n(2-store, 2-month sample -- illustrative only)")
ax.set_ylabel("SEK")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()


## 6. Takeaways

- **Top 3 parties by polling:** S (~33%), SD (~19%), M (~17-18%) — Sweden elects a parliament
  (Sept 13, 2026), not a president.

- **Food VAT (12% -> 6%) is already in effect** (Apr 2026-Dec 2027), and isn't really a contested
  campaign promise between the top 3 — S proposed it directly, and M/SD's governing coalition
  implemented it. Using this 2-month sample: food (12%-VAT) line items total **28.1M SEK excl. VAT**;
  at the historic 12% rate that's **31.5M SEK** consumer-facing, vs. **29.8M SEK** at 6%.
  - Pass-through scenario: consumers save ~**1.69M SEK** over 2 months on food in this sample
    (~5.4% price drop) -- a plausible demand tailwind Coop can't fully model without price-elasticity
    data (no VAT rate change occurred *within* this observed window).
  - Retain-margin scenario: if Coop held consumer prices flat instead, it could capture ~**1.59M SEK**
    of extra margin over 2 months in this sample (~5.7% revenue uplift on food) — a real, near-term
    margin lever regardless of which party governs, since the cut is already law through 2027.
  - The real political question is **what happens after Dec 2027** — a government review of a
    differentiated (up to 3-rate) food VAT is due by Dec 22, 2026, which could reshape this again.

- **Corporate tax is the sharper 3-way split**: S wants to hold at 20.6%, the government has already
  proposed cutting to 20.0%, and M wants to go lower still (unspecified exact target). Applied to
  this sample's gross margin (8.17M SEK, 2 months): the 20.6% -> 20.0% move alone is worth about
  **+49K SEK** retained in this sample — small in this illustrative scope, but directionally in
  Coop's favor under an M-led push, and a real point of tension since S remains the largest single
  party and is not proposing a cut.

- **Net read for Coop:** the food VAT cut is close to a done deal through 2027 regardless of
  election outcome (broad support), and gives Coop a real near-term choice between a demand play
  (pass savings to consumers) and a margin play (hold prices, keep the difference). Corporate tax is
  the genuinely contested lever, and given S's continued polling lead, a large near-term cut is not
  guaranteed even if M/SD form part of the next government -- worth treating any corporate-tax
  upside as a possibility to plan for, not a certainty to bank on.

- **What this notebook does NOT do**: forecast Coop's real national P&L, model consumer demand
  response to the VAT change, or account for coalition-formation dynamics after the election (a
  party's manifesto position doesn't guarantee it survives budget negotiations). All figures above
  are illustrative of mechanism and scale using real, cited party positions and real (if narrow)
  transaction data -- not a prediction.
